# 02 - Preprocessing

Steps:
- Stratified Random Split (train/test split)
- Drop Empty Columns
- Impute Gaps
- Drop Constant Columns
- Scale

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import RobustScaler

from src.preprocessing import split_secom, DropHighMissing, build_preprocessor

## 1. Stratified random split 

We want to keep the failure rate balanced between test and trial sets.

In [4]:
X_train, X_test, y_train, y_test = split_secom(test_size=0.20, random_state=42)

print(f'train: {len(X_train)} rows, {y_train.sum()} fails ({y_train.mean():.3f})')
print(f'test:  {len(X_test)} rows, {y_test.sum()} fails ({y_test.mean():.3f})')
print(f'# of columns: {X_train.shape[1]}')

train: 1253 rows, 83 fails (0.066)
test:  314 rows, 21 fails (0.067)
# of columns: 590


## 2. Drop columns that are mostly empty

First, we want to check if missingness relate to the label in any way. If not, we drop columns missing more than 50% of their values (*only on the training fold*). 

`DropHighMissing` learns which columns exceed the threshold in `fit`, then applies that samedecision to test.

In [ ]:
train_missing_frac = X_train.isna().mean()
high_missing = train_missing_frac[train_missing_frac > 0.50].index.tolist()

# for each high-missing column, check the failure rate among rows where that column is missing versus rows where it's present
diffs = {}
for col in high_missing:
    is_missing = X_train[col].isna()
    diffs[col] = y_train[is_missing].mean() - y_train[~is_missing].mean()

worst = max(diffs.values(), key=abs)
print(f"largest missing-vs-present fail-rate gap: {worst:.3f}")

largest missing-vs-present fail-rate gap: 0.012


The largest missing-vs-present fail-rate gap is only 0.012 (against a ~6.6% base rate), so whether these columns are missing or present barely changes the failure rate. 
Since these columns are also mostly empty, imputing would just fabricate most of their values so we can just drop them instead.

In [ ]:
drop_missing = DropHighMissing(threshold=0.5).fit(X_train)   

X_train_1 = drop_missing.transform(X_train)
X_test_1  = drop_missing.transform(X_test)                   

print(f'number of dropped columns: {len(drop_missing.columns_dropped_)}')
print(f'which columns are dropped: \n {drop_missing.columns_dropped_}')

number of dropped columns: 24
which columns are dropped: 
 ['sensor_85', 'sensor_109', 'sensor_110', 'sensor_111', 'sensor_157', 'sensor_158', 'sensor_220', 'sensor_244', 'sensor_245', 'sensor_246', 'sensor_292', 'sensor_293', 'sensor_358', 'sensor_382', 'sensor_383', 'sensor_384', 'sensor_492', 'sensor_516', 'sensor_517', 'sensor_518', 'sensor_578', 'sensor_579', 'sensor_580', 'sensor_581']


## 3. Impute the remaining gaps

Fills each gap with that column's **median** because mean is sensitive to outliers (computed on the training fold only). 

In [16]:
imputer = SimpleImputer(strategy='median').set_output(transform='pandas')
imputer.fit(X_train_1)                                  

X_train_2 = imputer.transform(X_train_1)
X_test_2  = imputer.transform(X_test_1)                 

print('missing values remaining (train):', int(X_train_2.isna().sum().sum()))
print('missing values remaining (test): ', int(X_test_2.isna().sum().sum()))
print(f'columns unchanged: {X_train_2.shape[1]}')

missing values remaining (train): 0
missing values remaining (test):  0
columns unchanged: 566


## 4. Drop constant (dead) columns

Check variance and drop columns that have a single value across every row

In [17]:
drop_constant = VarianceThreshold(threshold=0.0).set_output(transform='pandas')
drop_constant.fit(X_train_2)                                  # variances are measured on train set

X_train_3 = drop_constant.transform(X_train_2)
X_test_3  = drop_constant.transform(X_test_2)

n_dropped = X_train_2.shape[1] - X_train_3.shape[1]
print(f'dropped {n_dropped} constant columns')
print(f'columns: {X_train_2.shape[1]} -> {X_train_3.shape[1]}')

dropped 116 constant columns
columns: 566 -> 450


## 5. Scale

Using robust scaling over standard scaling because in a defect problem an extreme reading may be signal so we don't want a few outliers dominating the scaling.

In [18]:
scaler = RobustScaler().set_output(transform='pandas')
scaler.fit(X_train_3)                                  

X_train_pre = scaler.transform(X_train_3)
X_test_pre  = scaler.transform(X_test_3)

# peek at one column before vs after scaling
col = X_train_3.columns[0]
print(f'{col} before: median={X_train_3[col].median():.2f}, '
      f'range=[{X_train_3[col].min():.2f}, {X_train_3[col].max():.2f}]')
print(f'{col} after:  median={X_train_pre[col].median():.2f}, '
      f'range=[{X_train_pre[col].min():.2f}, {X_train_pre[col].max():.2f}]')
print(f'\nfinal shape: {X_train_pre.shape}')

sensor_0 before: median=3011.40, range=[2743.24, 3356.35]
sensor_0 after:  median=0.00, range=[-2.99, 3.84]

final shape: (1253, 450)
